In [ ]:
using BSplineKit
using LinearAlgebra
using IterativeSolvers
using ProgressMeter
using DelimitedFiles

In [163]:
function cheb(N)
    θ = range(0,length=N+1,stop=pi)
    x = reshape(-cos.(θ), N+1, 1)
    c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
    X = repeat(x, 1, N+1);
    dX = X - X';
    D = (c * (1 ./ c)') ./ (dX .+ I(N+1));
    D = D - diagm(vec(sum(D, dims=2)));
    return D
end


cheb (generic function with 1 method)

In [164]:
N = 1001
k = 10^-3
h = 10^-3
A = zeros(N,N)
r = k/(h^2)
u0 = [1;100 * ones(N-2,1);1]
for i = 1 : 1 : N
    if i == 1
        A[i,i] = 1
    elseif i == N
        A[i,i] = 1
    else
        A[i,i-1] = -r
        A[i,i] = 1 + 2 * r
        A[i,i+1] = -r
    end
end        

In [165]:
data = empty
@showprogress for i = 1 : 1 : 300
    u_temp = u0
    u0 = cg(A,u_temp)
    t = i * k * ones(N,1)
    data_temp = [t x u0]
    data = [data;data_temp]
end

In [166]:
data = data[2:end ,  : ]

300300×3 Matrix{Any}:
 0.001  0.0     1.0
 0.001  0.001   4.08155
 0.001  0.002   7.06717
 0.001  0.003   9.95987
 0.001  0.004  12.7625
 0.001  0.005  15.4779
 0.001  0.006  18.1088
 0.001  0.007  20.6578
 0.001  0.008  23.1275
 0.001  0.009  25.5203
 ⋮             
 0.3    0.992   1.1664
 0.3    0.993   1.1456
 0.3    0.994   1.12481
 0.3    0.995   1.10401
 0.3    0.996   1.08321
 0.3    0.997   1.06241
 0.3    0.998   1.04161
 0.3    0.999   1.0208
 0.3    1.0     1.0

In [167]:
writedlm("heat_convection.dat" , data ,'\t')